In [174]:
import cobrakbase
from cobrakbase.kbaseapi_cache import KBaseCache

In [175]:
kbase = KBaseCache(path='/scratch/fliu/data/kbase/cache')

In [176]:
%run /home/fliu/KE/easy_re.py
kb_re = init_re()

In [169]:
import subprocess
import pandas as pd
import os


def run_fastani(
    query_file,
    library_file="/home/fliu/KE/data/fastani_library.txt",
    output_file="/home/fliu/KE/data/fastani.out",
    threads=1,
    bin_fastani='/opt/FastANI/fastANI'
):
    cmd = [
        bin_fastani,
        "-t",
        str(threads),
        "--ql",
        query_file,
        "--rl",
        library_file,
        "-o",
        output_file,
    ]
    output = subprocess.run(
        cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True
    )
    fast_ani_result = None
    if os.path.exists(output_file):
        file_size = os.stat(output_file)
        if file_size.st_size > 0:
            fast_ani_result = pd.read_csv(output_file, header=None, sep="\t")
        return fast_ani_result, output
    return None, output


def get_contig_from_genome(genome):
    assembly = kbase.get_from_ws(genome.assembly_ref)
    return get_contig_from_assembly(assembly)


def get_contig_from_assembly(assembly, token):
    file_id = assembly.fasta_handle_ref
    file_path = "/home/fliu/KE/data/"
    res = kbase.download_file_from_kbase(token, file_id, file_path)
    return res


In [194]:
from tqdm import tqdm
from modelseedpy import MSGenome
from modelseedpy_ext.utils import sha_hex
def get_contig_set_hash(genome):
    h_seq = []
    for f in genome.features:
        s = f.seq.upper()
        _h = sha_hex(s)
        #genome_to_contig_h[g].add(_h)
        #if _h not in h_to_contig:
            #h_to_contig[_h] = set()
        #h_to_contig[_h].add(f.id)
        h_seq.append(sha_hex(s))
    hash_list = sorted(h_seq)
    hash_seq = "_".join(hash_list)
    hash_contig_set = sha_hex(hash_seq)
    return hash_contig_set
genome_to_h = {}
for genome_id, a in tqdm(assemblies.items()):
    fasta_handle_ref = a.fasta_handle_ref
    filename = f'/scratch/fliu/data/kbase/cache/handle/{fasta_handle_ref}'
    contigs = MSGenome.from_fasta(filename)
    genome_to_h[genome_id] = get_contig_set_hash(contigs)

100%|██████████| 450/450 [01:00<00:00,  7.48it/s]


In [195]:
g_metadata = {}
for genome_id in genomes:
    g_metadata[genome_id] = [genome_to_h[genome_id], assemblies[genome_id].fasta_handle_ref]
import json
with open('genome_h.json', 'w') as fh:
    fh.write(json.dumps(g_metadata))

In [192]:
len(assemblies)

450

In [8]:
assemblies = kbase.get_from_ws('155805/401/1')

In [17]:
ws_id = 155805
ws_os = kbase.list_objects(ws_id)

In [23]:
import os
genomes = {}
assemblies = {}
genomes_contigs = {}
for o in ws_os:
    if o[2].startswith('KBaseGenomes.Genome'):
        genome = kbase.get_from_ws(o[1], ws_id)
        assembly = kbase.get_from_ws(genome.assembly_ref)
        fasta_file = f'/scratch/fliu/data/kbase/cache/handle/{assembly.fasta_handle_ref}'
        genomes[genome.info.id] = genome
        assemblies[genome.info.id] = assembly
        genomes_contigs[genome.info.id] = fasta_file
        if not os.path.exists(fasta_file):
            kbase.download_file_from_kbase2(assembly.fasta_handle_ref, fasta_file)

In [171]:
print(len(genomes), len(assemblies))

310 310


In [177]:
ws_id = 163264
ws_os = kbase.list_objects(ws_id)
for o in ws_os:
    if o[2].startswith('KBaseGenomes.Genome'):
        if o[1] not in genomes:
            genome = kbase.get_from_ws(o[1], ws_id)
            assembly = kbase.get_from_ws(genome.assembly_ref)
            fasta_file = f'/scratch/fliu/data/kbase/cache/handle/{assembly.fasta_handle_ref}'
            genomes[genome.info.id] = genome
            assemblies[genome.info.id] = assembly
            genomes_contigs[genome.info.id] = fasta_file
            if not os.path.exists(fasta_file):
                kbase.download_file_from_kbase2(assembly.fasta_handle_ref, fasta_file)
        else:
            print('!!', o)

In [178]:
print(len(genomes), len(assemblies))

450 450


### Make ANI library

In [179]:
with open('/home/fliu/cliff_mags/ani_library.txt', 'w') as fh:
    for v in genomes_contigs.values():
        fh.write(f'{v}\n')

In [180]:
import time
t0 = time.perf_counter()
res = run_fastani(threads=120, 
                  output_file ='/home/fliu/cliff_mags/ani_self2.txt',
                  query_file  ='/home/fliu/cliff_mags/ani_library.txt',
                  library_file='/home/fliu/cliff_mags/ani_library.txt')
t1 = time.perf_counter()
print(t0, t1, t1 - t0)

6635140.992036197 6635318.379726824 177.38769062701613


In [181]:
import time
t0_gca = time.perf_counter()
res = run_fastani(threads=120, 
                  output_file ='/home/fliu/cliff_mags/ani_gtdb_gca2.txt',
                  query_file  ='/home/fliu/cliff_mags/ani_library.txt',
                  library_file='/scratch/fliu/data/gtdb/fastani_library_gca.txt')
t1_gca = time.perf_counter()
print('GCA', t0_gca, t1_gca, t1_gca - t0_gca)
import time
t0_gcf = time.perf_counter()
res = run_fastani(threads=120, 
                  output_file ='/home/fliu/cliff_mags/ani_gtdb_gcf2.txt',
                  query_file  ='/home/fliu/cliff_mags/ani_library.txt',
                  library_file='/scratch/fliu/data/gtdb/fastani_library_gcf.txt')
t1_gcf = time.perf_counter()
print('GCF', t0_gcf, t1_gcf, t1_gcf - t0_gcf)

GCA 6635318.419088311 6642069.15646193 6750.73737361934
GCF 6642069.157039954 6652293.266360494 10224.109320539981


In [182]:
print('GCA', t0_gca, t1_gca, t1_gca - t0_gca)
print('GCF', t0_gcf, t1_gcf, t1_gcf - t0_gcf)

GCA 6635318.419088311 6642069.15646193 6750.73737361934
GCF 6642069.157039954 6652293.266360494 10224.109320539981


In [29]:
print('GCA', t0_gca, t1_gca, t1_gca - t0_gca)
print('GCF', t0_gcf, t1_gcf, t1_gcf - t0_gcf)

GCA 1054252.238048342 1062300.094453059 8047.8564047168475
GCF 1062300.095004098 1074322.84910025 12022.754096152028


In [35]:
from modelseedpy.helpers import get_template, get_classifier
genome_classifier1 = get_classifier("knn_ACNP_RAST_filter_01_17_2023")
genome_classifier2 = get_classifier("knn_ACNP_RAST_full_01_17_2023")

In [36]:
genome_class = {}
for genome_id in genomes:
    genome = genomes[genome_id]
    genome_class1 = genome_classifier1.classify(genome)
    genome_class2 = genome_classifier2.classify(genome)
    genome_class[genome_id] = (genome_class1, genome_class2)

In [38]:
df_gtdb_class = pd.read_csv('gtdb_table.csv')
df_checkm_table = pd.read_csv('checkm_summary_table.tsv', sep='\t')

In [49]:
genome_check_m = {}
for row_id, d in df_checkm_table.iterrows():
    g_id = d['Bin Name']
    genome = genomes[g_id]
    genome_check_m[g_id] = (float(d['Completeness']), float(d['Contamination']))

In [70]:
genome_gtdb_clazz = {}
for row_id, d in df_gtdb_class.iterrows():
    g_id = assembly_id_to_genome[d['User Genome']]
    clazz = d['Classification']
    genome_gtdb_clazz[g_id] = clazz

In [81]:
for g_id, g in genomes.items():
    c1, c2 = genome_class[g_id]
    c_gtdb = genome_gtdb_clazz.get(g_id, '?')
    comp, cont = genome_check_m[g_id]
    a = assemblies[g_id]
    n_contigs = len(a.contigs)
    d = [g_id, c1, c2, c_gtdb, comp, cont, n_contigs]
    ss = '\t'.join([str(x) for x in d])
    print(ss)

Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.8.contigs__.RAST	N	N	d__Bacteria;p__Proteobacteria;c__Alphaproteobacteria;o__Rhodobacterales;f__Rhodobacteraceae;g__HIMB11;s__HIMB11 sp003486095	96.81	4.29	199
Salt_Pond_MetaG_R1_B_D1_MG_DASTool_bins_metabat.20.contigs__.RAST	N	N	d__Bacteria;p__Proteobacteria;c__Alphaproteobacteria;o__Rhodobacterales;f__Rhodobacteraceae;g__IMCC34051;s__	98.48	3.11	133
Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_concoct_out.4.contigs__.RAST	N	C	d__Bacteria;p__Planctomycetota;c__Planctomycetes;o__Pirellulales;f__Pirellulaceae;g__Crateriforma;s__	89.73	3.6	664
Salt_Pond_MetaG_R2_restored_DShore_MG_DASTool_bins_concoct_out.46.contigs__.RAST	N	N	d__Bacteria;p__Spirochaetota;c__Spirochaetia;o__Spirochaetales_C;f__PWMO01;g__;s__	98.67	5.73	129
Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_concoct_out.21.contigs__.RAST	A	A	?	98.37	2.61	89
Salt_Pond_MetaG_R1_C_D1_MG_DASTool_bins_metabat.31.contigs__.RAST	A	A	?	82.03	4.16	509
Salt_Pond_MetaG_R2_A_H2O_MG_DASTool_bins_concoct

In [83]:
genome = genomes['Salt_Pond_MetaG_R2A_C_D2_MG_DASTool_bins_metabat.10.contigs__.RAST']
genome_class1 = genome_classifier1.classify(genome)
genome_class2 = genome_classifier2.classify(genome)


In [101]:
genome_classifier1.model.n_neighbors

5

In [102]:
genome_classifier2.model.n_neighbors

5

In [84]:
print(genome_class1, genome_class2)

N C


In [103]:
import json
gtdb_gca = None
with open('/scratch/fliu/data/gtdb/metadata_gca.json', 'r') as fh:
    gtdb_gca = json.load(fh)
gtdb_gcf = None
with open('/scratch/fliu/data/gtdb/metadata_gcf.json', 'r') as fh:
    gtdb_gcf = json.load(fh)

In [109]:
import hashlib
from modelseedpy import MSGenome
genome_assembly_h = {}
for k, v in genomes_contigs.items():
    genome = MSGenome.from_fasta(v)
    contig_set = [f.seq.upper() for f in genome.features]
    hash_list = []
    for dna_seq in contig_set:
        symbols = {}
        for o in dna_seq:
            if o not in symbols:
                symbols[o] = 0
            symbols[o] += 1
        h = kb_re.dna_store.store_sequence(dna_seq)
        hash_list.append(h)
    hash_list = sorted(hash_list)
    hash_seq = "_".join(hash_list)
    hash_contig_set = hashlib.sha256(hash_seq.encode("utf-8")).hexdigest()
    genome_assembly_h[k] = hash_contig_set

In [111]:
gtdb_h_collition = {}
for g, h in genome_assembly_h.items():
    match = set()
    if h in gtdb_gca['h_to_genome']:
        match |= set(gtdb_gca['h_to_genome'][h])
        #print('GCA', g, gtdb_gca['h_to_genome'][h])
    if h in gtdb_gcf['h_to_genome']:
        match |= set(gtdb_gcf['h_to_genome'][h])
        #print('GCF', g, gtdb_gcf['h_to_genome'][h])
    gtdb_h_collition[g] = match

In [113]:
df_gtdb_meta_ar = pd.read_csv('/scratch/fliu/data/gtdb/ar53_metadata_r214.tsv', sep='\t')
df_gtdb_meta_bac = pd.read_csv('/scratch/fliu/data/gtdb/bac120_metadata_r214.tsv', sep='\t')

In [114]:
gtdb_metadata = {}
for row_id, d in df_gtdb_meta_ar.iterrows():
    k = d['accession'][3:]
    if k not in gtdb_metadata:
        gtdb_metadata[k] = d
    else:
        print('!')
for row_id, d in df_gtdb_meta_bac.iterrows():
    k = d['accession'][3:]
    if k not in gtdb_metadata:
        gtdb_metadata[k] = d
    else:
        print('!')

In [121]:
handle_to_genome_id= {}
for a, h in genomes_contigs.items():
    if h not in handle_to_genome_id:
        handle_to_genome_id[h] = a
    else:
        print('!!')

In [122]:
fast_ani_self = pd.read_csv('/home/fliu/cliff_mags/ani_self.txt', header=None, sep="\t")
scores = {}
for row_id, d in fast_ani_self.iterrows():
    v = d[2]
    g1 = handle_to_genome_id[d[0]]
    g2 = handle_to_genome_id[d[1]]
    if g1 not in scores:
        scores[g1] = {}
    scores[g1][g2] = v

In [127]:
df_ani = pd.read_csv('/home/fliu/cliff_mags/ani_gtdb_gca.txt', header=None, sep="\t")
for row_id, d in df_ani.iterrows():
    v = d[2]
    g1 = handle_to_genome_id[d[0]]
    h = d[1].split('/')[-1][:-4]
    g2_list = gtdb_gca['h_to_genome'][h]
    if g1 not in scores:
        scores[g1] = {}
    for _g in g2_list:
        _p = _g.split('_')
        _g_id = _p[0] + '_' + _p[1]
        #d = gtdb_metadata[_g_id]
        scores[g1][_g_id] = v
df_ani = pd.read_csv('/home/fliu/cliff_mags/ani_gtdb_gcf.txt', header=None, sep="\t")
for row_id, d in df_ani.iterrows():
    v = d[2]
    g1 = handle_to_genome_id[d[0]]
    h = d[1].split('/')[-1][:-4]
    g2_list = gtdb_gcf['h_to_genome'][h]
    if g1 not in scores:
        scores[g1] = {}
    for _g in g2_list:
        _p = _g.split('_')
        _g_id = _p[0] + '_' + _p[1]
        #d = gtdb_metadata[_g_id]
        scores[g1][_g_id] = v

In [131]:
ignore_gtdb = set()
all_genomes = []
for genome in genomes:
    if genome not in all_genomes:
        all_genomes.append(genome)
    #comp = clades_comp[genome]
    genome_ani = scores[genome]
    for other_g in genome_ani:
        v = genome_ani[other_g]
        #print(genome, v, v>90, other_g)
        if v > 90 and other_g not in all_genomes and other_g not in ignore_gtdb:
            all_genomes.append(other_g)

In [132]:
import numpy as np
_size = len(all_genomes)
mat = np.zeros((_size, _size))

In [134]:
for i1 in range(_size):
    g1 = all_genomes[i1]
    for i2 in range(_size):
        g2 = all_genomes[i2]
        if g1 == g2:
            mat[i1][i2] = 100.0
        if g1 in scores and g2 in scores[g1]:
            mat[i1][i2] = scores[g1][g2]
# fill symtrical values
for i1 in range(_size):
    for i2 in range(_size):
        v = mat[i1][i2]
        if v != 0.0 and mat[i2][i1] == 0:
            mat[i2][i1] = v

In [136]:
h = ['', '', '', ''] + all_genomes
print('\t'.join(h))
for i1 in range(_size):
    l = all_genomes[i1]
    comp = None
    t_ = 'V'
    _n = l
    if l in gtdb_metadata:
        d = gtdb_metadata[l]
        comp = d['checkm_completeness']
        t_ = d['ncbi_genome_category']
        _n = d['ncbi_strain_identifiers']
    else:
        comp = genome_check_m.get(l, [0, 0])[0]
        t_ = "KBase"
        #print(l)
        pass
    row = [_n, t_, str(comp), l]
    row += [str(x) for x in mat[i1]]
    print('\t'.join(row))

				Salt_Pond_MetaGSF2_B_H2O_MG_DASTool_bins_metabat.8.contigs__.RAST	GCA_017859495.1	GCA_017852835.1	GCA_017859535.1	GCA_008081275.1	GCA_017859635.1	GCA_003486095.1	GCA_010032605.1	GCA_010028145.1	GCA_010023165.1	GCA_023258295.1	GCA_010028055.1	GCA_010023135.1	GCA_010023435.1	GCA_010023665.1	GCA_010022345.1	GCA_010022375.1	GCA_010027975.1	GCA_010024165.1	GCA_010023555.1	GCA_913063195.1	GCA_018222665.1	GCA_023260575.1	GCA_000472185.1	GCA_022424725.1	Salt_Pond_MetaG_R1_B_D1_MG_DASTool_bins_metabat.20.contigs__.RAST	Salt_Pond_MetaG_R2_B_D2_MG_DASTool_bins_concoct_out.4.contigs__.RAST	Salt_Pond_MetaG_R2_restored_DShore_MG_DASTool_bins_concoct_out.46.contigs__.RAST	Salt_Pond_MetaG_R1_A_D2_MG_DASTool_bins_concoct_out.21.contigs__.RAST	GCA_020055655.1	Salt_Pond_MetaG_R1_C_D1_MG_DASTool_bins_metabat.31.contigs__.RAST	GCF_900101245.1	GCF_900111485.1	GCF_004217335.1	GCF_000337475.1	GCF_005938085.1	Salt_Pond_MetaG_R2_A_H2O_MG_DASTool_bins_concoct_out.29.contigs__.RAST	GCF_000427685.1	Salt_Pond_M

In [149]:
feature_id_to_genome = {}
ignored = {}
with open('/scratch/fliu/data/cliff/all_prot.fna', 'w') as fh:
    for genome_id, genome in genomes.items():
        ignored[genome_id] = set()
        for f in genome.features:
            if f.seq:
                if f.id not in feature_id_to_genome:
                    feature_id_to_genome[f.id] = genome_id
                    fh.write(f'>{f.id}\n')
                    fh.write(f'{f.seq.upper()}\n')
                else:
                    ignored[genome_id].add(f.id)
            else:
                ignored[genome_id].add(f.id)

In [150]:
from modelseedpy_ext.mmseqs.mmseqs import MMseqs2

In [151]:
for genome_id, genome in genomes.items():
    genome.id = genome_id

In [159]:
mmseqs2 = MMseqs2(list(genomes.values()), '/scratch/fliu/data/cliff/')

In [161]:
mmseqs2.mmseqs_bin = '/opt/MMseqs2.build/src/mmseqs'
res = mmseqs2.cluster('/scratch/fliu/data/cliff/all_prot.fna')

['/opt/MMseqs2.build/src/mmseqs', 'easy-cluster', '--threads', '320', '--min-seq-id', '0.0', '--cov-mode', '0', '-c', '0.8', '/scratch/fliu/data/cliff/all_prot.fna', '/scratch/fliu/data/cliff//output/mmseqs', '/scratch/fliu/data/cliff//output']


In [185]:
def read_ani_out(f):
    ani_score = {}
    with open(f, 'r') as fh:
        l = fh.readline()
        while(l):
            _p = l.split('\t')
            g1, g2, score, k1, k2 = _p
            if g1 not in ani_score:
                ani_score[g1] = {}
            ani_score[g1][g2] = (score, k1, k2)
            l = fh.readline()
    return ani_score
ani_score_gca = read_ani_out('ani_gtdb_gca2.txt')
ani_score_gcf = read_ani_out('ani_gtdb_gcf2.txt')

In [186]:
len(ani_score_gca), len(ani_score_gcf) #(237, 151)

(348, 230)

In [187]:
foreign_genomes = set()
global_ani = {}
for k1 in ani_score_gca:
    g1 = k1.split('/')[-1]
    if g1 not in global_ani:
        global_ani[g1] = {}
    for k2 in ani_score_gca[k1]:
        g2 = k2.split('/')[-1].split('.fna')[0]
        global_ani[g1][g2] = ani_score_gca[k1][k2]
        foreign_genomes.add(g2)
for k1 in ani_score_gcf:
    g1 = k1.split('/')[-1]
    if g1 not in global_ani:
        global_ani[g1] = {}
    for k2 in ani_score_gcf[k1]:
        g2 = k2.split('/')[-1].split('.fna')[0]
        global_ani[g1][g2] = ani_score_gcf[k1][k2]
        foreign_genomes.add(g2)

In [189]:
len(foreign_genomes)

21727

In [190]:
%store foreign_genomes

Stored 'foreign_genomes' (set)
